In [117]:
import os
import re
from collections import defaultdict, Counter
from pathlib import Path

import numpy as np
import pandas as pd

import nltk
from nltk.stem import PorterStemmer, SnowballStemmer

# 0. NLTK SETUP

try:
    nltk.corpus.stopwords.words('english')
except LookupError:
    nltk.download('stopwords')

In [118]:
# 1. PREPROCESSING

def preprocess_text(text: str,
                    stem_mode: str = "Porter Stemmer",
                    normalize: bool = False) -> list:
    """
    Preprocess text:
      - remove <s>, </s>
      - tokenize via regex
      - lowercase
      - remove stopwords
      - stemming (Porter or Snowball)
    """
    # Remove special tokens
    text = re.sub(r"</?s>", " ", text)

    # Tokenizer
    tokenizer = nltk.RegexpTokenizer(r'(?:[A-Za-z]\.)+|[A-Za-z]+[-@]\d+(?:\.\d+)?|\d+[A-Za-z]+|\d+(?:[.,-]\d+)?%?|\w+(?:[-/]\w+)*|[.!?]+')

    tokens = tokenizer.tokenize(text)

    # Lowercase
    tokens = [t.lower() for t in tokens]

    # # Remove stopwords
    # stop = set(nltk.corpus.stopwords.words('english'))
    # tokens = [t for t in tokens if t not in stop and len(t) > 1]

    # # Stemming
    # if stem_mode == "Porter Stemmer":
    #     stemmer = PorterStemmer()
    # elif stem_mode == "Snowball Stemmer":
    #     stemmer = SnowballStemmer("english")
    # else:
    #     stemmer = PorterStemmer()

    # tokens = [stemmer.stem(t) for t in tokens]

    # normalize flag unused for now, but kept for compatibility
    return tokens


In [119]:
#read this foldes 
# Attribute Data:
# Folder: Attributes/
# Description: This folder contains 18 attribute files. Each file includes a set of related word sequences.

def read_attributes(folder: str) -> dict:
    """
    Read attribute files from the specified folder.
    Each file is expected to contain a list of words or phrases.
    Returns a dictionary where keys are filenames and values are lists of words.
    """
    attributes = {}
    folder_path = Path(folder)

    # Read all files that start with "Attribute_" (no .txt extension)
    for file_path in folder_path.glob("Attribute_*"):
        if file_path.is_file() and not file_path.name.startswith('.'):
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read().strip()
                words = content.splitlines()
                #attribute_1 => 1 is the num_attribute
                match = re.match(r"Attribute_(\d+)", file_path.stem)
                if match:
                    attribute_num = match.groups()[0]
                    attributes[attribute_num] = [word.strip() for word in words if word.strip()]

    return attributes

In [120]:
#read articles
#Extract article_id and volume from filename: Article_<id>_Volume_<v> in folder ..\All-in-many
def read_articles(folder: str) -> pd.DataFrame:
    """
    Read articles from the specified folder.
    Each file is expected to be named in the format Article_<id>_Volume_<v>.
    Returns a DataFrame with columns: article_id, volume, and content.
    """
    articles = []
    folder_path = Path(folder)

    # Read all files that start with "Article_" (no .txt extension)
    for file_path in folder_path.glob("Article_*"):
        if file_path.is_file() and not file_path.name.startswith('.'):
            match = re.match(r"Article_(\d+)_Volume_(\d+)", file_path.name)
            if match:
                article_id, volume = match.groups()
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read().strip()
                    articles.append({
                        'article_id': int(article_id),
                        'volume': int(volume),
                        'content': content
                    })

    return pd.DataFrame(articles)


In [121]:
def build_feature_matrix(articles_df: pd.DataFrame,
                         attributes: dict,
                         normalize: bool = False,
                         stem_mode: str = "Porter Stemmer") -> pd.DataFrame:
    """
    Build a feature matrix where each row is an article and each column is an attribute file.
    The value for attribute j is the total number of occurrences of all word-sequences
    listed in attribute j within the article, weighted by sequence length.
    
    For example: 'swarm' counts as 1, 'particle swarm' counts as 2 (length of 2 words).

    If normalize=True, both article text and attribute sequences are tokenized, lowercased,
    stopwords removed and stemmed according to `stem_mode` before matching. Matches require
    contiguous token sequence equality. If normalize=False, raw (lowercased) substring
    matches are counted (overlapping matches included).
    
    The weight is always based on the number of words in the ORIGINAL (non-normalized) sequence.
    """
    def count_seq_in_tokens(article_tokens, seq_tokens):
        if not seq_tokens or not article_tokens:
            return 0
        m, n = len(seq_tokens), len(article_tokens)
        if m > n:
            return 0
        count = 0
        # sliding window equality for contiguous token matches
        for i in range(n - m + 1):
            if article_tokens[i:i + m] == seq_tokens:
                count += 1
        return count

    rows = []
    attr_names = list(attributes.keys())

    for _, row in articles_df.iterrows():
        article_id = row.get('article_id')
        volume = row.get('volume')
        content = row.get('content', "") or ""
        feature_vals = {}

        if normalize:
            article_tokens = preprocess_text(content, stem_mode=stem_mode, normalize=False)
            # precompute preprocessed attribute sequences (list of token lists)
            preproc_attr_seqs = {}
            for aname in attr_names:
                seqs = attributes.get(aname, [])
                # Store both preprocessed tokens and original word count
                preproc_attr_seqs[aname] = [
                    (preprocess_text(s, stem_mode=stem_mode, normalize=False), len(s.split()))
                    for s in seqs
                ]
            # count occurrences by token-matching, weighted by original sequence word count
            for aname, seq_data_list in preproc_attr_seqs.items():
                total = 0
                for seq_tokens, original_word_count in seq_data_list:
                    occurrences = count_seq_in_tokens(article_tokens, seq_tokens)
                    # Weight by the number of words in the ORIGINAL (non-normalized) sequence
                    total += occurrences * original_word_count
                feature_vals[aname] = total

        else:
            content_lower = content.lower()
            for aname in attr_names:
                total = 0
                for seq in attributes.get(aname, []):
                    seq_lower = seq.lower()
                    # count overlapping occurrences using lookahead
                    if seq_lower.strip() == "":
                        continue
                    pattern = re.compile(r'(?={})'.format(re.escape(seq_lower)))
                    occurrences = len(pattern.findall(content_lower))
                    # Weight by the number of words in the original sequence
                    num_words = len(seq.split())
                    total += occurrences * num_words
                feature_vals[aname] = total

        out_row = {'article_id': article_id, 'volume': volume}
        out_row.update(feature_vals)
        rows.append(out_row)

    feature_df = pd.DataFrame(rows, columns=['article_id', 'volume'] + attr_names)
    return feature_df


In [122]:
# Example usage (execute in a cell after reading attributes and articles):
attrs = read_attributes("../Attributes")

# Check how many attributes were read
print(f"Number of attributes read: {len(attrs)}")
print(f"Attribute keys: {sorted(attrs.keys())}")

#verify one attribute file (keys are strings)
print(f"\nAttribute '2' contents:")
print(attrs['2'])

Number of attributes read: 18
Attribute keys: ['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '2', '3', '4', '5', '6', '7', '8', '9']

Attribute '2' contents:
['swarm', 'swarm intelligence', 'particle swarm optimization', 'pso', 'particle', 'ant colony optimization', 'aco', 'colony', 'artificial bee colony', 'abc', 'artificial bee', 'firefly algorithm', 'fa', 'firefly', 'grey wolf optimizer', 'gwo', 'bat algorithm', 'ba', 'cuckoo search', 'cs', 'whale optimization algorithm', 'woa', 'harris hawks optimization', 'hho', 'harris hawks', 'slime mould algorithm', 'sma', 'sine cosine algorithm', 'sca', 'moth-flame optimizer', 'mfo', 'sparrow search algorithm', 'ssa', 'grey wolf optimisation', 'gwo', 'simulated annealing', 'sa']


In [123]:
articles = read_articles("../All-in-many")   # adjust path as needed
#verify one article
print(articles)

     article_id  volume                                            content
0           100      18  Comparative analysis of accuracy and computati...
1           101      18  An efficient multilevel thresholding image seg...
2           102      18  Optimising SMIB system stability: FOPID contro...
3           103      18  Evolutionary multitasking algorithm based on a...
4           104      18  An efficient plant disease prediction model ba...
..          ...     ...                                                ...
112          96      18  Machine learning with industrial robots: explo...
113          97      18  Modified random-oppositional chaotic artificia...
114          98      18  Student psychology based optimization algorith...
115          99      18  Fast implementation of extreme learning machin...
116           9      18  Quasi-opposition-based learning in a shuffled ...

[117 rows x 3 columns]


In [124]:
X = build_feature_matrix(articles, attrs, normalize=True, stem_mode="Porter Stemmer")
X.head()

,article_id,volume,1,10,11,12,13,14,15,16,17,18,2,3,4,5,6,7,8,9
0,100,18,7,0,0,0,6,0,0,0,0,0,23,0,0,0,10,0,1,2
1,101,18,6,0,0,0,0,0,4,0,0,0,7,0,2,0,8,0,0,0
2,102,18,4,0,0,0,0,0,0,0,0,0,17,0,0,0,2,0,0,0
3,103,18,1,0,0,0,0,0,0,0,0,0,0,4,0,0,2,0,2,0
4,104,18,0,9,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0


In [125]:
# Verify the feature matrix dimensions and statistics
print("=" * 70)
print("FEATURE MATRIX SUMMARY")
print("=" * 70)
print(f"\nShape: {X.shape}")
print(f"Number of instances (articles): {len(X)}")
print(f"Number of features (attributes): {X.shape[1] - 2}")  # exclude article_id and volume
print(f"\nFeature columns: {sorted([col for col in X.columns if col not in ['article_id', 'volume']])}")

print("\n" + "=" * 70)
print("FEATURE STATISTICS")
print("=" * 70)

# Get feature columns only (exclude article_id and volume)
feature_cols = [col for col in X.columns if col not in ['article_id', 'volume']]

# Statistics for each attribute
for attr in sorted(feature_cols, key=lambda x: int(x)):
    total_count = X[attr].sum()
    non_zero_articles = (X[attr] > 0).sum()
    max_count = X[attr].max()
    mean_count = X[attr].mean()
    print(f"Attribute {attr:2s}: Total occurrences={total_count:4d}, "
          f"Non-zero articles={non_zero_articles:3d}, "
          f"Max={max_count:3d}, Mean={mean_count:.2f}")

print("\n" + "=" * 70)
print("SAMPLE INSTANCES")
print("=" * 70)
print("\nFirst 10 instances:")
print(X.head(10).to_string(index=False))

FEATURE MATRIX SUMMARY

Shape: (117, 20)
Number of instances (articles): 117
Number of features (attributes): 18

Feature columns: ['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '2', '3', '4', '5', '6', '7', '8', '9']

FEATURE STATISTICS
Attribute 1 : Total occurrences= 314, Non-zero articles= 77, Max= 14, Mean=2.68
Attribute 2 : Total occurrences= 274, Non-zero articles= 31, Max= 32, Mean=2.34
Attribute 3 : Total occurrences=  90, Non-zero articles= 27, Max=  6, Mean=0.77
Attribute 4 : Total occurrences= 160, Non-zero articles= 49, Max= 16, Mean=1.37
Attribute 5 : Total occurrences=  30, Non-zero articles= 11, Max=  6, Mean=0.26
Attribute 6 : Total occurrences= 148, Non-zero articles= 47, Max= 10, Mean=1.26
Attribute 7 : Total occurrences=   4, Non-zero articles=  2, Max=  2, Mean=0.03
Attribute 8 : Total occurrences=  45, Non-zero articles= 27, Max=  5, Mean=0.38
Attribute 9 : Total occurrences=  12, Non-zero articles=  8, Max=  3, Mean=0.10
Attribute 10: Total occurrenc

In [126]:
# sort rows by article_id (and volume as tie-breaker) and reorder attribute columns numerically
sorted_attrs = sorted(feature_cols, key=lambda x: int(x))
cols_order = ['article_id', 'volume'] + sorted_attrs
X = X.sort_values(['article_id', 'volume']).reset_index(drop=True)[cols_order]

# display the first rows to verify
X.head()

,article_id,volume,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,1,18,5,0,0,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0
1,2,18,0,0,0,0,0,0,0,0,0,5,7,0,3,4,0,4,0,0
2,3,18,4,2,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0
3,4,18,5,0,3,0,6,3,0,1,0,0,0,0,0,0,0,0,0,0
4,5,18,1,0,2,2,2,0,0,1,1,0,0,0,1,0,0,0,0,0


In [127]:
# save to csv file
X.to_csv("instances.csv", index=False)
